In [4]:
# ==============================
# ATTENDANCE + EMOTION SYSTEM (Kaggle-ready)
# ==============================

import os
import cv2
import joblib
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model


#  student dataset
STUDENT_TRAIN_DIR = "/kaggle/input/student-faces/students"   # <-- your path

# FER2013 (emotion dataset)
FER_TRAIN_DIR = "/kaggle/input/fer2013/train"
FER_TEST_DIR  = "/kaggle/input/fer2013/test"

ATTENDANCE_INPUT_DIR = "/kaggle/input/student-faces/students"  


ATT_START = datetime.time(9, 30)
ATT_END   = datetime.time(10, 0)
ENFORCE_TIME_WINDOW = False  

SVM_PATH = "face_recognition_svm.pkl"
LE_PATH  = "label_encoder.pkl"
EMO_PATH = "emotion_model.h5"

# Face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def load_student_faces(base_dir):
    if not os.path.isdir(base_dir):
        raise FileNotFoundError(f"Student directory not found: {base_dir}")

    X, y = [], []
    class_names = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])
    if not class_names:
        raise ValueError(f"No class folders found inside: {base_dir}")

    for person in class_names:
        person_dir = os.path.join(base_dir, person)
        for fname in os.listdir(person_dir):
            fpath = os.path.join(person_dir, fname)
            if not os.path.isfile(fpath):
                continue
            img = cv2.imread(fpath)
            if img is None:
                continue
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # Detect faces; if none found, use the center crop as fallback
            faces = face_cascade.detectMultiScale(gray, 1.1, 4)
            if len(faces) == 0:
                # fallback: use the whole image resized (not ideal, but keeps pipeline robust)
                face = cv2.resize(gray, (100, 100))
                X.append(face.flatten())
                y.append(person)
            else:
                for (x, y1, w, h) in faces:
                    face = gray[y1:y1+h, x:x+w]
                    face = cv2.resize(face, (100, 100))
                    X.append(face.flatten())
                    y.append(person)
    return np.array(X), np.array(y), class_names


def train_face_id_model():
    print(" Training SVM face-ID model from:", STUDENT_TRAIN_DIR)
    X, y, class_names = load_student_faces(STUDENT_TRAIN_DIR)
    print(f"  Loaded faces: {X.shape}, classes: {len(class_names)} -> {class_names}")

    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    svm = SVC(kernel="linear", probability=True)
    svm.fit(X, y_enc)

    joblib.dump(svm, SVM_PATH)
    joblib.dump(le, LE_PATH)
    print("Saved:", SVM_PATH, LE_PATH)


def train_emotion_model():
    print("Training Emotion model from FER2013...")
    datagen = ImageDataGenerator(rescale=1./255)

    train_gen = datagen.flow_from_directory(
        FER_TRAIN_DIR, target_size=(48,48), color_mode="grayscale",
        batch_size=32, class_mode="categorical"
    )
    val_gen = datagen.flow_from_directory(
        FER_TEST_DIR, target_size=(48,48), color_mode="grayscale",
        batch_size=32, class_mode="categorical"
    )

    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32,(3,3),activation="relu",input_shape=(48,48,1)),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Conv2D(64,(3,3),activation="relu"),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128,activation="relu"),
        tf.keras.layers.Dense(train_gen.num_classes,activation="softmax")
    ])
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

    model.fit(train_gen, validation_data=val_gen, epochs=10, verbose=1)
    model.save(EMO_PATH)


    emotion_labels = list(train_gen.class_indices.keys())
    joblib.dump(emotion_labels, "emotion_labels.pkl")
    print("Saved:", EMO_PATH, "and emotion_labels.pkl")


def within_window(now_t, start_t, end_t):
    return (start_t <= now_t <= end_t)

def run_attendance(input_dir=ATTENDANCE_INPUT_DIR):
    if not os.path.isdir(input_dir):
        raise FileNotFoundError(f"Attendance input dir not found: {input_dir}")

    # Load models
    svm = joblib.load(SVM_PATH)
    le  = joblib.load(LE_PATH)
    emo_model = load_model(EMO_PATH)
    emotion_labels = joblib.load("emotion_labels.pkl")

    # Prepare attendance dict
    attendance = {name: {"Status":"Absent", "Emotion":"-", "Time":"-"} for name in le.classes_}

    # Check time
    now = datetime.datetime.now()
    now_t = now.time()
    allow = (within_window(now_t, ATT_START, ATT_END) or (not ENFORCE_TIME_WINDOW))
    if not allow:
        print(f"Current time {now_t.strftime('%H:%M:%S')} is outside {ATT_START}-{ATT_END}. "
              f"Set ENFORCE_TIME_WINDOW=False to test.")
        # Still save a blank report listing all students as Absent
        df = pd.DataFrame.from_dict(attendance, orient="index")
        df.to_csv("attendance_report.csv")
        print("Saved 'attendance_report.csv' (all Absent due to time window).")
        return

    # Iterate images
    for root, _, files in os.walk(input_dir):
        for fname in files:
            if not fname.lower().endswith((".jpg",".jpeg",".png",".bmp")):
                continue
            fpath = os.path.join(root, fname)
            img = cv2.imread(fpath)
            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 4)

            # Process each detected face
            if len(faces) == 0:
                # fallback: try the whole image
                roi = cv2.resize(gray, (100,100)).flatten().reshape(1,-1)
                pred = svm.predict(roi)[0]
                name = le.inverse_transform([pred])[0]
                emo_in = cv2.resize(gray, (48,48)).reshape(1,48,48,1)/255.0
                emo_pred = np.argmax(emo_model.predict(emo_in, verbose=0), axis=1)[0]
                emo = emotion_labels[emo_pred]
                attendance[name] = {
                    "Status": "Present",
                    "Emotion": emo,
                    "Time": now.strftime("%Y-%m-%d %H:%M:%S")
                }
            else:
                for (x, y1, w, h) in faces:
                    face_gray = gray[y1:y1+h, x:x+w]
                    id_in = cv2.resize(face_gray, (100,100)).flatten().reshape(1,-1)
                    pred = svm.predict(id_in)[0]
                    name = le.inverse_transform([pred])[0]

                    emo_in = cv2.resize(face_gray, (48,48)).reshape(1,48,48,1)/255.0
                    emo_pred = np.argmax(emo_model.predict(emo_in, verbose=0), axis=1)[0]
                    emo = emotion_labels[emo_pred]

                    attendance[name] = {
                        "Status": "Present",
                        "Emotion": emo,
                        "Time": now.strftime("%Y-%m-%d %H:%M:%S")
                    }

    # Save CSV
    df = pd.DataFrame.from_dict(attendance, orient="index")
    df.index.name = "Student"
    df.to_csv("attendance_report.csv")
    print("Saved 'attendance_report.csv'")
    print(df)



train_face_id_model()

train_emotion_model()

run_attendance(ATTENDANCE_INPUT_DIR)


 Training SVM face-ID model from: /kaggle/input/student-faces/students
  Loaded faces: (38, 10000), classes: 8 -> ['aadhila', 'akhil', 'aswathy', 'drisya', 'farhana', 'greena', 'manoj', 'mini']
Saved: face_recognition_svm.pkl label_encoder.pkl
Training Emotion model from FER2013...
Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.
Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


898/898 ━━━━━━━━━━━━━━━━━━━━ 41s 43ms/step - accuracy: 0.3086 - loss: 1.7282 - val_accuracy: 0.4450 - val_loss: 1.4580
Epoch 2/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 37s 41ms/step - accuracy: 0.4606 - loss: 1.4198 - val_accuracy: 0.4731 - val_loss: 1.3624
Epoch 3/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.5430 - loss: 1.2111 - val_accuracy: 0.5192 - val_loss: 1.2580
Epoch 5/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.5677 - loss: 1.1372 - val_accuracy: 0.5189 - val_loss: 1.2675
Epoch 6/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.6006 - loss: 1.0640 - val_accuracy: 0.5258 - val_loss: 1.2665
Epoch 7/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 33s 37ms/step - accuracy: 0.6336 - loss: 0.9914 - val_accuracy: 0.5311 - val_loss: 1.2668
Epoch 8/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.6585 - loss: 0.9157 - val_accuracy: 0.5340 - val_loss: 1.2764
Epoch 9/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.6844 - loss: 0.8560 - val_accurac